# Phân tích bất thường CPU theo thời gian

Notebook dùng cùng pipeline với giao diện và API: đọc CSV, khảo sát dữ liệu, lấy 70% đầu để huấn luyện và dự đoán 30% cuối bằng MAD, Isolation Forest, LOF, One-Class SVM. Mọi thống kê và mô hình chỉ được fit trên tập huấn luyện.

Tệp `cpu_0.csv` đi kèm chỉ có 19 dòng. Đây là ví dụ chạy thử quy trình, chưa đủ để kết luận chất lượng phát hiện trên dữ liệu thực tế. Không có kết nối cơ sở dữ liệu hay giám sát thời gian thực.


In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

# Hỗ trợ mở từ workspace root, thư mục dự án hoặc notebooks/.
cwd = Path.cwd().resolve()
candidates = []
for base in (cwd, *cwd.parents):
    candidates.extend((base, base / "ml-anomaly-detection", base / "DACN" / "ml-anomaly-detection"))
ROOT = next((candidate for candidate in candidates
             if (candidate / "src" / "pipeline.py").is_file()
             and (candidate / "data" / "cpu_0.csv").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Không tìm thấy dự án ml-anomaly-detection và data/cpu_0.csv từ thư mục hiện tại.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.data_prep import add_nab_labels, load_timeseries
from src.eda import seasonality_figure, trend_figure
from src.pipeline import run_detection

print("Thư mục dự án:", ROOT)


## 1. Đọc CSV và kiểm tra phạm vi chú giải

Hai mốc NAB trong `add_nab_labels` chỉ áp dụng cho chuỗi EC2 CPU tương ứng. Chúng là chú giải từng điểm, chưa phải cửa sổ bất thường hoặc cách chấm điểm NAB chính thức. Đoạn mẫu đi kèm không chứa hai mốc này; nhãn `1` ở đây chỉ biểu thị không trùng các mốc chú giải đã cung cấp, chưa phải xác nhận độc lập rằng mọi điểm đều bình thường.


In [ ]:
source = ROOT / "data" / "cpu_0.csv"
df = add_nab_labels(load_timeseries(source, value_col="value"))
print(f"Số dòng: {len(df)}; thời gian: {df['timestamp'].min()} đến {df['timestamp'].max()}")
print("Số điểm trùng chú giải NAB:", int((df["is_anomaly"] == -1).sum()))
df.head()


## 2. Khảo sát mô tả

Biểu đồ xu hướng và trung bình theo lịch chỉ mô tả những điểm quan sát được. Với một đoạn dữ liệu ngắn, chúng chưa chứng minh tính mùa vụ và không tham gia huấn luyện mô hình ở bên dưới.


In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(df["timestamp"], df["value"], marker=".", label="Giá trị quan sát")
annotated = df.loc[df["is_anomaly"] == -1]
if not annotated.empty:
    ax.scatter(annotated["timestamp"], annotated["value"], color="red", label="Điểm chú giải NAB")
ax.set(title="CPU: đoạn dữ liệu quan sát", xlabel="Thời gian", ylabel="Giá trị")
ax.legend()
fig.tight_layout()
plt.show()


In [ ]:
trend_plot = trend_figure(df, "value")
trend_plot


In [ ]:
seasonality_plot = seasonality_figure(df, "value")
seasonality_plot


## 3. Huấn luyện trên quá khứ, dự đoán phần còn lại

`run_detection` chia dữ liệu theo thứ tự thời gian. Scaler, median/MAD và cả ba mô hình học máy chỉ fit phần huấn luyện. Nhãn chỉ dùng để đánh giá sau dự đoán, không có trong đặc trưng. Khi MAD bằng 0, pipeline dùng ngưỡng dung sai fit từ train và xét cả tăng lẫn giảm bất thường.

`contamination=0.01` là cấu hình độ nhạy cho các mô hình liên quan; nó không buộc 1% tập test phải là bất thường. MAD dùng ngưỡng robust z-score 3.5.


In [ ]:
run = run_detection(df, label_col="is_anomaly", train_fraction=0.7, contamination=0.01)
split = run.metadata["split_index"]
print(f"Huấn luyện: {run.metadata['train_rows']} dòng; kiểm tra: {run.metadata['test_rows']} dòng.")
warning_text = {
    "Small dataset: this run verifies functionality and cannot establish general detection quality.":
        "Dữ liệu ít: lần chạy này chỉ kiểm tra quy trình, chưa đủ kết luận chất lượng phát hiện.",
    "Test labels contain no anomalies: recall/F1 cannot support a detection-quality conclusion.":
        "Tập kiểm tra không có điểm trùng chú giải bất thường: chưa thể dùng recall/F1 để kết luận chất lượng.",
    "Rows with missing, nonnumeric or infinite metric values were removed before the chronological split; see cleaning counts.":
        "Các dòng thiếu giá trị, không phải số hoặc vô hạn đã được loại trước khi chia train/test; xem metadata cleaning.",
}
for warning in run.metadata["warnings"]:
    print("Lưu ý:", warning_text.get(warning, warning))


## 4. Xem các cảnh báo trên tập kiểm tra

Chỉ những dòng sau mốc chia mới có dự đoán. `-1` là cảnh báo bất thường, `1` là bình thường theo mô hình; cảnh báo chưa phải xác nhận sự cố. Số cảnh báo ít hơn không đồng nghĩa mô hình tốt hơn. Điểm số lớn hơn biểu thị bình thường hơn trong từng mô hình; không so trực tiếp điểm số giữa các thuật toán.


In [ ]:
comparison = run.predictions.copy()
alert_counts = pd.DataFrame({
    "thuật_toán": run.metadata["model_names"],
    "số_cảnh_báo_trên_test": [int((comparison[f"{name}_prediction"] == -1).sum())
                             for name in run.metadata["model_names"]],
    "số_dòng_test": len(comparison),
})
print(alert_counts.to_string(index=False))
comparison.head(10)


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 7), sharex=True)
for ax, name in zip(axes.flat, run.metadata["model_names"]):
    ax.plot(df["timestamp"], df["value"], color="lightgray", label="Toàn bộ dữ liệu")
    ax.plot(comparison["timestamp"], comparison["value"], marker=".", label="Tập kiểm tra")
    alerts = comparison.loc[comparison[f"{name}_prediction"] == -1]
    ax.scatter(alerts["timestamp"], alerts["value"], color="red", label="Cảnh báo mô hình")
    ax.axvline(run.frame.iloc[split]["timestamp"], color="black", linestyle="--", label="Bắt đầu tập kiểm tra")
    ax.set(title=name, ylabel="Giá trị")
    ax.legend(fontsize=8)
fig.autofmt_xdate()
fig.tight_layout()
plt.show()


## 5. Đánh giá khi có đủ nhãn

Precision, recall và F1 là chỉ số từng điểm trên tập kiểm tra, không phải điểm NAB chính thức. Khi tập test không chứa điểm bất thường được chú giải, notebook không hiển thị bảng xếp hạng F1 để tránh diễn giải sai các giá trị 0. Cần dữ liệu dài hơn và chú giải phù hợp trước khi so sánh chất lượng mô hình.


In [ ]:
test_annotation_count = int((comparison["is_anomaly"] == -1).sum())
quality_supported = test_annotation_count > 0
if quality_supported:
    metrics_table = pd.DataFrame.from_dict(run.metrics, orient="index")
    print(metrics_table.to_string())
    print("Các chỉ số chỉ mô tả tập kiểm tra hiện tại; không suy rộng thành chất lượng thực tế.")
else:
    metrics_table = pd.DataFrame(columns=["precision", "recall", "f1"])
    print("Chưa hiển thị precision/recall/F1: tập test không chứa điểm bất thường được chú giải.")
    print("Hãy dùng dữ liệu đủ dài có nhãn độc lập; số cảnh báo hiện tại chỉ minh họa đầu ra của mô hình.")
